# HOS for Dummies — Interactive 3-D Demo

**What this notebook teaches:**
- Why the ordinary FFT cannot distinguish a *cracked shaft* from a *misaligned shaft*
- What the **bispectrum** and **trispectrum** measure that the FFT misses
- How to read the 3-D Plotly surfaces and ball plots (Sinha 2007 Figs. 5–8 style)

**No signal-processing background required.** Every concept is introduced with an analogy before the math.

---

## The key idea in one sentence

> A FFT tells you **what frequencies are present**. Higher-order spectra tell you **whether those frequencies are talking to each other** — i.e., whether their phases are locked together across time.

A breathing crack in a rotating shaft bends the rotor slightly open and closed once per revolution. This creates a nonlinear interaction: the vibration at 1× rotation speed *generates* components at 2×, 3×, and their phases are locked together. Misalignment also creates harmonics, but through a different mechanism — the phase locking pattern is different. The bispectrum and trispectrum reveal this difference; the FFT alone cannot.

In [17]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from signal_utils import bispectrum, bicoherence, trispectrum, amplitude_spectrum
from plot_utils import (
    plotly_time,
    plotly_spectrum,
    plotly_bispectrum_surface,
    plotly_bicoherence_surface,
    plotly_trispectrum_balls,
)

# Sinha §3.3 parameters (same as constants.py)
FS   = 2560.0   # Hz — sampling rate
T    = 25.0     # s  — record length (T_LONG)
t    = np.arange(0, T, 1 / FS)

# Sinha's 750 RPM case: 1X = 12.5 Hz
F1X  = 12.5     # Hz
F2X  = 2 * F1X  # 25 Hz
F3X  = 3 * F1X  # 37.5 Hz

print(f"Record: {T} s · {FS} Hz = {len(t):,} samples")
print(f"1X = {F1X} Hz  |  2X = {F2X} Hz  |  3X = {F3X} Hz")

Record: 25.0 s · 2560.0 Hz = 64,000 samples
1X = 12.5 Hz  |  2X = 25.0 Hz  |  3X = 37.5 Hz


---
## Part 1 — Two signals that look identical in the FFT

We will build two signals:

| Signal | Story | Phase relationship |
|--------|-------|-------------------|
| `x_crack` | Cracked shaft — harmonics generated by the same nonlinear mechanism | Phases are **locked**: φ(2X) = 2·φ(1X), φ(3X) = 3·φ(1X) |
| `x_noise` | Independent excitations — each harmonic has a random, unrelated phase | Phases are **independent** across measurement blocks |

Both signals have **exactly the same amplitude at 1X, 2X, 3X**. Their FFT magnitude spectra are identical. Only the HOS can tell them apart.

In [18]:
phi = 0.8   # arbitrary base phase

# --- CRACK-LIKE: fixed phase coupling ---
x_crack = (
    1.0 * np.cos(2 * np.pi * F1X * t)
    + 0.5 * np.cos(2 * np.pi * F2X * t + 2 * phi)    # φ(2X) = 2·φ(1X)
    + 0.3 * np.cos(2 * np.pi * F3X * t + 3 * phi)    # φ(3X) = 3·φ(1X)
)

# --- INDEPENDENT PHASES: randomized per segment (no coupling) ---
rng    = np.random.default_rng(7)
nfft_b = int(round(FS / 1.25))          # 2048 — one block per bispectrum segment
t_blk  = np.arange(nfft_b) / FS
n_blk  = int(np.ceil(len(t) / nfft_b))
x_noise = np.concatenate([
    (
        1.0 * np.cos(2 * np.pi * F1X * t_blk + rng.uniform(0, 2 * np.pi))
        + 0.5 * np.cos(2 * np.pi * F2X * t_blk + rng.uniform(0, 2 * np.pi))
        + 0.3 * np.cos(2 * np.pi * F3X * t_blk + rng.uniform(0, 2 * np.pi))
    )
    for _ in range(n_blk)
])[:len(t)]

print("Signals created.")
print(f"x_crack  — max={x_crack.max():.3f}  std={x_crack.std():.3f}")
print(f"x_noise  — max={x_noise.max():.3f}  std={x_noise.std():.3f}")

Signals created.
x_crack  — max=1.545  std=0.818
x_noise  — max=1.742  std=0.819


### 1a — Time domain: they look the same

In [19]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("x_crack (phase-coupled harmonics)",
                                    "x_noise (independent random phases)"))
t_show = t[t <= 0.4]   # first 0.4 s
fig.add_trace(go.Scatter(x=t_show, y=x_crack[:len(t_show)], mode="lines",
                         line=dict(color="firebrick", width=1.2), name="crack"), row=1, col=1)
fig.add_trace(go.Scatter(x=t_show, y=x_noise[:len(t_show)], mode="lines",
                         line=dict(color="steelblue", width=1.2), name="noise"), row=2, col=1)
fig.update_xaxes(title_text="Time (s)", row=2, col=1)
fig.update_yaxes(title_text="Amplitude")
fig.update_layout(title="Time domain — visually similar", height=450, showlegend=False)
fig.show()

### 1b — Amplitude spectrum: identical peaks at 1X, 2X, 3X

The FFT sees the same three peaks regardless of the phase relationship. This is why FFT-based diagnostics struggle to separate crack from misalignment.

In [20]:
amp_c, freqs_amp = amplitude_spectrum(x_crack, FS)
amp_n, _         = amplitude_spectrum(x_noise, FS)

FMAX_SPEC = 60.0
idx = freqs_amp <= FMAX_SPEC

fig = go.Figure()
fig.add_trace(go.Scatter(x=freqs_amp[idx], y=amp_c[idx], mode="lines",
                         name="x_crack", line=dict(color="firebrick", width=2)))
fig.add_trace(go.Scatter(x=freqs_amp[idx], y=amp_n[idx], mode="lines",
                         name="x_noise", line=dict(color="steelblue", width=1.5,
                                                    dash="dash")))
for f_harm, label in [(F1X, "1X"), (F2X, "2X"), (F3X, "3X")]:
    fig.add_vline(x=f_harm, line_dash="dot", line_color="grey", opacity=0.5,
                  annotation_text=label, annotation_position="top")
fig.update_layout(title="Amplitude spectrum — peaks are identical for both signals",
                  xaxis_title="Frequency (Hz)", yaxis_title="Amplitude",
                  legend=dict(x=0.7, y=0.95))
fig.show()

---
## Part 2 — Bispectrum: the phase-coupling detector

### How the bispectrum works

For every frequency pair $(f_l, f_m)$, the bispectrum computes:

$$B(f_l, f_m) = \langle X(f_l)\, X(f_m)\, X^*(f_l+f_m) \rangle$$

Think of it this way: multiply the Fourier coefficient at $f_l$ by the one at $f_m$, then multiply by the *conjugate* of the coefficient at their sum $f_l+f_m$.  Average this product over many short segments of the signal.

- If the phases at $f_l$, $f_m$, $f_l+f_m$ are **randomly related** across segments, the product cancels out → $B \approx 0$
- If the phases are **locked together** (one is generated from the others), the product always points the same direction → $|B| \gg 0$

Only the **non-redundant triangle** $f_l \le f_m$, $f_l + f_m \le f_{Nyquist}$ contains unique information.

In [21]:
print("Computing bispectra (this takes ~10 s for 25-s records) ...")
B_crack, b2_crack, freqs = bispectrum(x_crack, FS)
B_noise, b2_noise, _     = bispectrum(x_noise, FS, nfft=nfft_b, noverlap=0)
print(f"Done. Frequency resolution: {freqs[1]-freqs[0]:.4f} Hz")

Computing bispectra (this takes ~10 s for 25-s records) ...
Done. Frequency resolution: 1.2500 Hz


### 2a — Bispectrum surface: crack-like signal

**What to look for:**  
Peaks at the frequency pairs where phase coupling exists:
- $(f_l, f_m) = (1\text{X}, 1\text{X})$: the pair that sums to $2\text{X}$
- $(f_l, f_m) = (1\text{X}, 2\text{X})$: the pair that sums to $3\text{X}$

In Sinha's paper these appear as two distinct peaks in Figs. 5–6.

In [22]:
fig = plotly_bispectrum_surface(
    B_crack, freqs, fmax_hz=50.0,
    title="Bispectrum |B|/max|B| — crack-like signal (compare Sinha Fig. 5)"
)

# Annotate the expected peak locations
peak_pairs = [(F1X, F1X, "(1X,1X)→2X"), (F1X, F2X, "(1X,2X)→3X")]
i_f  = int(np.argmin(np.abs(freqs - F1X)))
i_2f = int(np.argmin(np.abs(freqs - F2X)))
B_max = np.abs(B_crack).max()
for fl, fm, label in peak_pairs:
    il = int(np.argmin(np.abs(freqs - fl)))
    im = int(np.argmin(np.abs(freqs - fm)))
    z_val = np.abs(B_crack[il, im]) / B_max
    fig.add_trace(go.Scatter3d(
        x=[fl], y=[fm], z=[z_val + 0.05],
        mode="text", text=[label],
        textfont=dict(color="cyan", size=11),
        showlegend=False,
    ))

fig.show()

### 2b — Bispectrum surface: independent-phase signal

**What to look for:**  
The surface should be flat / near-zero everywhere.  Because phases are random and uncorrelated between segments, the bispectrum averages to zero.

In [23]:
fig = plotly_bispectrum_surface(
    B_noise, freqs, fmax_hz=50.0,
    title="Bispectrum — independent-phase signal (no coupling → flat surface)"
)
fig.show()

---
## Part 3 — Bicoherence²: the normalized version

The raw bispectrum $|B|$ depends on signal amplitude, so a louder signal always looks more coupled. The **squared bicoherence** $b^2$ fixes this by dividing out the power:

$$b^2(f_l, f_m) = \frac{|B(f_l, f_m)|^2}{\langle |X(f_l) X(f_m)|^2 \rangle \cdot \langle |X(f_l+f_m)|^2 \rangle} \in [0, 1]$$

**$b^2 = 1$** means the phases are perfectly locked (100% of the power at $f_l+f_m$ is generated by the interaction of $f_l$ and $f_m$).  
**$b^2 = 0$** means the phases are completely independent.

This is the quantity that Sinha uses to compare different operating conditions (Figs. 5–6 in his paper show $|B|/\max|B|$, but the shape is the same as $b^2$).

In [24]:
print(f"Crack-like signal — peak b² at (1X,1X): {b2_crack[i_f, i_f]:.4f}")
print(f"Crack-like signal — peak b² at (1X,2X): {b2_crack[i_f, i_2f]:.4f}")
print(f"Independent-phase — max  b²:             {b2_noise.max():.4f}")

Crack-like signal — peak b² at (1X,1X): 1.0000
Crack-like signal — peak b² at (1X,2X): 1.0000
Independent-phase — max  b²:             0.0846


In [25]:
fig_c = plotly_bicoherence_surface(
    b2_crack, freqs, fmax_hz=50.0,
    title="Bicoherence² b² — crack-like signal  (peaks at 1X–1X and 1X–2X)"
)
fig_c.show()

In [26]:
fig_n = plotly_bicoherence_surface(
    b2_noise, freqs, fmax_hz=50.0,
    title="Bicoherence² b² — independent-phase signal  (flat → no coupling)"
)
fig_n.show()

---
## Part 4 — Trispectrum: third-order coupling in 3-D

The **trispectrum** extends the idea to three frequency arguments:

$$T(f_l, f_m, f_n) = \langle X^*(f_l)\, X^*(f_m)\, X^*(f_n)\, X(f_l+f_m+f_n) \rangle$$

It is a **3-D object** — the result lives in $(f_l, f_m, f_n)$ space.  We plot it as balls where each ball represents one frequency triplet $(f_l, f_m, f_n)$ and the **ball size** encodes the normalized amplitude $|T|/\max|T|$.

For the crack-like signal with harmonics at 1X, 2X, 3X:
- The dominant peak is at $(1\text{X}, 1\text{X}, 1\text{X})$, which sums to $3\text{X}$ — the cubic phase coupling

Sinha plots these as spheres in Figs. 7 and 8.

In [27]:
print("Computing trispectrum (this may take ~30 s) ...")
T_dict, freqs_t = trispectrum(x_crack, FS, threshold=0.10, fmax_hz=50.0)
print(f"Non-zero entries above 0.10 threshold: {len(T_dict)}")
T_abs = {k: abs(v) for k, v in T_dict.items()}
T_max = max(T_abs.values())
for k in sorted(T_abs, key=T_abs.get, reverse=True):
    fl, fm, fn = freqs_t[k[0]], freqs_t[k[1]], freqs_t[k[2]]
    print(f"  T({fl:.2f}, {fm:.2f}, {fn:.2f}) Hz — |T|/max = {T_abs[k]/T_max:.3f}")

Computing trispectrum (this may take ~30 s) ...
Non-zero entries above 0.10 threshold: 4
  T(12.50, 12.50, 12.50) Hz — |T|/max = 1.000
  T(11.25, 12.50, 13.75) Hz — |T|/max = 0.250
  T(11.25, 12.50, 12.50) Hz — |T|/max = 0.250
  T(12.50, 12.50, 13.75) Hz — |T|/max = 0.250


In [28]:
fig = plotly_trispectrum_balls(
    T_dict, freqs_t, fmax_hz=50.0, amp_min=0.10,
    title="Trispectrum — crack-like signal (compare Sinha Fig. 7)"
)
fig.show()

---
## Part 5 — Side-by-side comparison: effect of SNR

In a real machine, the probe signal is noisy. Here we add 40 dB SNR Gaussian noise (Sinha §5 setting) to the crack-like signal and check whether the bispectrum peaks survive.

In [29]:
from signal_utils import add_awgn

x_noisy = add_awgn(x_crack, snr_db=40.0, rng=np.random.default_rng(99))

print("Computing bispectrum of noisy signal ...")
B_noisy, b2_noisy, _ = bispectrum(x_noisy, FS)
print(f"b²(1X,1X) = {b2_noisy[i_f, i_f]:.4f}  (clean: {b2_crack[i_f, i_f]:.4f})")
print(f"b²(1X,2X) = {b2_noisy[i_f, i_2f]:.4f}  (clean: {b2_crack[i_f, i_2f]:.4f})")

Computing bispectrum of noisy signal ...
b²(1X,1X) = 1.0000  (clean: 1.0000)
b²(1X,2X) = 1.0000  (clean: 1.0000)


In [30]:
fig = plotly_bicoherence_surface(
    b2_noisy, freqs, fmax_hz=50.0,
    title=f"Bicoherence² — crack + 40 dB AWGN  (Sinha §5 noise level)"
)
fig.show()

---
## Summary

| What you measured | Signal | 3-D plot | Key finding |
|---|---|---|---|
| Bispectrum surface | crack-like | Surface peaks at (1X,1X) and (1X,2X) | Phase coupling is present |
| Bispectrum surface | independent-phases | Flat surface | No coupling |
| Bicoherence² | crack-like | b²≈1 at coupled pairs | Coupling is 100% (perfect synthetic case) |
| Bicoherence² | independent-phases | b²≈0 everywhere | No coupling confirmed |
| Trispectrum balls | crack-like | Ball at (1X,1X,1X) | Cubic coupling at 1X |
| Bicoherence² | crack + 40 dB noise | Peaks still visible | HOS robust to realistic SNR |

### Why this matters for the thesis

Sinha (2007) showed experimentally that:
- A **cracked shaft** produces bispectrum peaks at (1X,1X) and (1X,2X) because the breathing crack nonlinearly generates 2X from 1X.
- A **misaligned shaft** also produces harmonics, but with a different phase-coupling topology — the bispectrum peak pattern is different.

The next sprints (02–04) will replicate this with ROSS-simulated probe signals using the Mayes crack model, and then compute the HOS using this pipeline to reproduce Sinha's Figs. 5–10.